# HighwayEnv Intersection PPO Creeping

This notebook trains and compares PPO agents entirely inside `highway-env` on the intersection task.

The goal is not just crossing fast. The creeping agent should use continuous throttle as a communication signal: approach the conflict area, slow into a low-moderate assertion speed, avoid blasting through the intersection, and still complete the route with a decent success rate.

Compared agents:

- `PPO_standard_fast_dense`: non-creeping PPO trained with a fast-crossing reward.
- `PPO_creeping_learned_dense`: PPO policy network with unconstrained continuous throttle, warm-started from traffic-aware demonstrations and scored with reward-only negotiation shaping.
- reference policies: fast constant throttle and a traffic-aware creeping teacher used only for curriculum, not as a runtime controller.

Primary performance metrics: success rate, average collisions, collision rate, and time to collision. Behavior metrics: mean speed in the conflict zone, high-speed conflict-zone entry rate, creep-speed rate, and throttle smoothness.

## Current Result Snapshot

The current local same-seed evaluation used 80 episodes on denser HighwayEnv intersection traffic (`8` initial vehicles, `0.45` spawn probability):

| agent | success | collision | avg collisions | mean TTC on collision | conflict speed | high-speed zone rate | creep-speed rate |
|---|---:|---:|---:|---:|---:|---:|---:|
| PPO_standard_fast_dense | 0.538 | 0.463 | 0.463 | 4.86 s | 10.40 m/s | 0.988 | 0.000 |
| PPO_creeping_learned_dense | 0.663 | 0.338 | 0.338 | 8.10 s | 3.76 m/s | 0.230 | 0.666 |
| scripted_non_creeping_fast_dense | 0.538 | 0.463 | 0.463 | 4.82 s | 10.03 m/s | 0.988 | 0.000 |
| traffic_aware_teacher_dense | 0.663 | 0.325 | 0.325 | 10.24 s | 2.96 m/s | 0.141 | 0.733 |

So the learned creeping policy improves success by `+0.125`, reduces collision rate by `-0.125`, and makes the conflict-zone behavior visibly different: much slower entry, much less high-speed entry, and longer time before collisions when they happen.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import torch
except Exception:
    torch = None

try:
    from IPython.display import display, Video
except Exception:
    display = print
    Video = None

WORKSPACE = Path.cwd()
while WORKSPACE.name != "agv_Unstructured_Labs" and WORKSPACE.parent != WORKSPACE:
    WORKSPACE = WORKSPACE.parent

NOTEBOOK_DIR = WORKSPACE / "notebooks"
RESULTS_DIR = NOTEBOOK_DIR / "results" / "intersectionPPOCreeping"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from highway_ppo_creeping_utils import (
    ConstantThrottlePolicy,
    HighwayIntersectionConfig,
    ScriptedCreepingPolicy,
    TrafficAwareCreepingPolicy,
    behavior_clone_ppo_actor,
    evaluate_agent,
    evaluate_reference_policies,
    make_intersection_env,
    train_ppo,
)

DEVICE = "cuda" if torch is not None and torch.cuda.is_available() else "cpu"
CPU_CORES = os.cpu_count() or 1

print(f"Workspace: {WORKSPACE}")
print(f"Results:   {RESULTS_DIR}")
print(f"Device:    {DEVICE}")
print(f"CPU cores: {CPU_CORES}")

## Training Settings

`FORCE_RETRAIN=False` means the notebook reuses trained models if they already exist. Set it to `True` to train again.

There are no runtime action constraints or residual throttle priors in the main learned creeping policy. The action is raw continuous HighwayEnv throttle. The training curriculum is:

1. train a standard PPO route-completion/fast-crossing policy,
2. expose compact negotiation cues in observation space: TTC, ego y-position, conflict-zone flags, and reward target speed,
3. distill traffic-aware creeping demonstrations into the PPO actor to avoid the local optimum of either rushing or parking,
4. evaluate the neural PPO policy directly with no teacher and no hard action override.

The reward shaping still determines what is considered good behavior: safe probing, speed-conditioned progress, no parking, commit after a safe gap, collision penalty, and arrival bonus.

In [ ]:
RUN_TRAINING = True
FORCE_RETRAIN = False
RUN_EVALUATION = True
RUN_REFERENCES = True

N_ENVS = 4
VEC_BACKEND = "dummy"  # use "subproc" outside notebooks if you want to push CPU parallelism harder
SEED = 51
DENSE_EVAL_CFG = HighwayIntersectionConfig(initial_vehicle_count=8, spawn_probability=0.45)

STANDARD_TIMESTEPS = 12_000
TEACHER_EPISODES = 100
BC_EPOCHS = 80
EVAL_EPISODES = 80
EVAL_SEED = 31_000

STANDARD_MODEL_PATH = RESULTS_DIR / "highway_ppo_standard_features_12k.zip"
CREEPING_MODEL_PATH = RESULTS_DIR / "highway_ppo_creeping_bc_features.zip"

print(STANDARD_MODEL_PATH)
print(CREEPING_MODEL_PATH)

## Train PPO Agents

Both agents use continuous `highway-env` throttle (`ContinuousAction`, longitudinal only). The standard PPO learns a fast crossing objective. The creeping PPO actor is then warm-started from traffic-aware demonstrations, but the final evaluated policy is just the neural PPO actor producing throttle.

In [ ]:
if RUN_TRAINING:
    if FORCE_RETRAIN or not STANDARD_MODEL_PATH.exists():
        train_ppo(
            "standard",
            total_timesteps=STANDARD_TIMESTEPS,
            model_path=STANDARD_MODEL_PATH,
            n_envs=N_ENVS,
            seed=SEED,
            device=DEVICE,
            vec_backend=VEC_BACKEND,
            use_features=True,
        )
    else:
        print(f"Using existing standard PPO model: {STANDARD_MODEL_PATH}")

    if FORCE_RETRAIN or not CREEPING_MODEL_PATH.exists():
        teacher = TrafficAwareCreepingPolicy()
        creeping_model, bc_history = behavior_clone_ppo_actor(
            base_model_path=STANDARD_MODEL_PATH,
            teacher=teacher,
            model_path=CREEPING_MODEL_PATH,
            episodes=TEACHER_EPISODES,
            seed=26_000,
            epochs=BC_EPOCHS,
            device=DEVICE,
        )
        bc_history.to_csv(RESULTS_DIR / "highway_ppo_creeping_bc_history.csv", index=False)
    else:
        print(f"Using existing creeping PPO model: {CREEPING_MODEL_PATH}")

## Evaluate

The evaluation uses the same dense-traffic seed set for every agent so the comparison is fair. The main behavior separation should appear in `mean_creep_zone_speed`, `high_speed_zone_rate`, and `creep_speed_rate`, while success rate rises and collision rate falls.

In [ ]:
from stable_baselines3 import PPO

combined_eval_df = pd.DataFrame()
combined_summary_df = pd.DataFrame()

if RUN_EVALUATION:
    standard_model = PPO.load(STANDARD_MODEL_PATH, device=DEVICE)
    creeping_model = PPO.load(CREEPING_MODEL_PATH, device=DEVICE)

    frames = []
    summaries = []

    eval_specs = [
        (standard_model, "PPO_standard_fast_dense", "standard"),
        (creeping_model, "PPO_creeping_learned_dense", "creeping_negotiation"),
    ]

    for model, name, variant in eval_specs:
        df, summary = evaluate_agent(model, name, variant, episodes=EVAL_EPISODES, seed=EVAL_SEED, cfg=DENSE_EVAL_CFG, use_features=True)
        frames.append(df)
        summaries.append(summary)

    if RUN_REFERENCES:
        ref_specs = [
            (ConstantThrottlePolicy(0.35), "scripted_non_creeping_fast_dense", "standard"),
            (TrafficAwareCreepingPolicy(), "traffic_aware_teacher_dense", "creeping_negotiation"),
        ]
        ref_frames = []
        ref_summaries = []
        for policy, name, variant in ref_specs:
            df, summary = evaluate_agent(policy, name, variant, episodes=EVAL_EPISODES, seed=EVAL_SEED, cfg=DENSE_EVAL_CFG, use_features=True)
            ref_frames.append(df)
            ref_summaries.append(summary)
        ref_df = pd.concat(ref_frames, ignore_index=True)
        ref_summary = pd.concat(ref_summaries, ignore_index=True)
        frames.append(ref_df)
        summaries.append(ref_summary)

    combined_eval_df = pd.concat(frames, ignore_index=True)
    combined_summary_df = pd.concat(summaries, ignore_index=True)

    combined_eval_df.to_csv(RESULTS_DIR / "highway_final_dense_eval_episodes.csv", index=False)
    combined_summary_df.to_csv(RESULTS_DIR / "highway_final_dense_performance_summary.csv", index=False)

    display(
        combined_summary_df[
            [
                "agent",
                "episodes",
                "success_rate",
                "collision_rate",
                "average_collisions",
                "mean_time_to_collision_s",
                "mean_survival_time_s",
                "mean_creep_zone_speed",
                "high_speed_zone_rate",
                "creep_speed_rate",
                "mean_abs_throttle",
            ]
        ]
    )

## Dashboard

The first plot checks task performance. The second checks whether the behavior actually changed from non-creeping to creeping.

In [ ]:
import matplotlib.pyplot as plt

summary_path = RESULTS_DIR / "highway_final_dense_performance_summary.csv"
summary = pd.read_csv(summary_path) if summary_path.exists() else combined_summary_df.copy()

if summary.empty:
    print("No evaluation summary yet. Run the evaluation cell first.")
else:
    display(summary)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    summary.plot.bar(x="agent", y="success_rate", ax=axes[0], legend=False, color="#2a9d8f")
    summary.plot.bar(x="agent", y="collision_rate", ax=axes[1], legend=False, color="#e76f51")
    summary.plot.bar(x="agent", y="mean_time_to_collision_s", ax=axes[2], legend=False, color="#457b9d")
    axes[0].set_title("Success Rate")
    axes[1].set_title("Collision Rate")
    axes[2].set_title("Mean Time To Collision")
    axes[0].set_ylim(0, 1)
    axes[1].set_ylim(0, 1)
    axes[2].set_ylabel("seconds")
    for ax in axes:
        ax.tick_params(axis="x", rotation=35)
        ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    plt.show()

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    summary.plot.bar(x="agent", y="mean_creep_zone_speed", ax=axes[0], legend=False, color="#264653")
    summary.plot.bar(x="agent", y="high_speed_zone_rate", ax=axes[1], legend=False, color="#f4a261")
    summary.plot.bar(x="agent", y="creep_speed_rate", ax=axes[2], legend=False, color="#2a9d8f")
    axes[0].set_title("Conflict-Zone Speed")
    axes[0].set_ylabel("m/s")
    axes[1].set_title("High-Speed Zone Rate")
    axes[1].set_ylim(0, 1)
    axes[2].set_title("Creep-Speed Rate")
    axes[2].set_ylim(0, 1)
    for ax in axes:
        ax.tick_params(axis="x", rotation=35)
        ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    plt.show()

## Metric Delta

This cell computes the learned creeping policy's change relative to the non-creeping PPO baseline.

## Optional Rendering

Set `RENDER_ONE_EPISODE=True` to record one HighwayEnv episode. Use `MODEL_TO_RENDER="creeping"` to inspect the creeping PPO behavior or `"standard"` for the non-creeping PPO.

In [ ]:
summary_path = RESULTS_DIR / "highway_final_dense_performance_summary.csv"
if summary_path.exists():
    summary = pd.read_csv(summary_path).set_index("agent")
    base = summary.loc["PPO_standard_fast_dense"]
    learned = summary.loc["PPO_creeping_learned_dense"]
    delta = pd.DataFrame(
        {
            "metric": [
                "success_rate",
                "collision_rate",
                "average_collisions",
                "mean_time_to_collision_s",
                "mean_creep_zone_speed",
                "high_speed_zone_rate",
                "creep_speed_rate",
            ],
            "standard": [
                base["success_rate"],
                base["collision_rate"],
                base["average_collisions"],
                base["mean_time_to_collision_s"],
                base["mean_creep_zone_speed"],
                base["high_speed_zone_rate"],
                base["creep_speed_rate"],
            ],
            "learned_creeping": [
                learned["success_rate"],
                learned["collision_rate"],
                learned["average_collisions"],
                learned["mean_time_to_collision_s"],
                learned["mean_creep_zone_speed"],
                learned["high_speed_zone_rate"],
                learned["creep_speed_rate"],
            ],
        }
    )
    delta["absolute_delta"] = delta["learned_creeping"] - delta["standard"]
    display(delta)
else:
    print("Run evaluation first to create highway_final_dense_performance_summary.csv")

In [ ]:
RENDER_ONE_EPISODE = False
MODEL_TO_RENDER = "creeping"  # "standard" or "creeping"

if RENDER_ONE_EPISODE:
    from gymnasium.wrappers import RecordVideo

    if MODEL_TO_RENDER == "standard":
        model = PPO.load(STANDARD_MODEL_PATH, device=DEVICE)
        variant = "standard"
    else:
        model = PPO.load(CREEPING_MODEL_PATH, device=DEVICE)
        variant = "creeping_negotiation"

    video_dir = RESULTS_DIR / "videos"
    video_dir.mkdir(parents=True, exist_ok=True)
    env = make_intersection_env(variant=variant, seed=EVAL_SEED, render_mode="rgb_array", cfg=DENSE_EVAL_CFG, use_features=True)
    env = RecordVideo(env, video_folder=str(video_dir), name_prefix=f"{MODEL_TO_RENDER}_episode")
    obs, info = env.reset(seed=EVAL_SEED)
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        done = bool(terminated or truncated)
    env.close()

    videos = sorted(video_dir.glob(f"{MODEL_TO_RENDER}_episode*.mp4"))
    if videos:
        print(videos[-1])
        if Video is not None:
            display(Video(str(videos[-1]), embed=True))

## Interpretation

A useful creeping agent should not merely avoid collisions by stopping. Look for all four properties together:

- success rate rises over the non-creeping PPO,
- collision rate and average collisions fall,
- mean time to collision increases when collisions still happen,
- high-speed conflict-zone entry collapses while creep-speed rate rises.

### What worked

The final learned policy is `PPO_creeping_learned_dense`. It is evaluated with raw continuous throttle; there is no residual action prior, speed cap, scripted controller, or action override. Compared to `PPO_standard_fast_dense`, the learned creeping policy:

- raises success rate from `0.5375` to `0.6625`,
- lowers collision rate from `0.4625` to `0.3375`,
- lowers average collisions from `0.4625` to `0.3375`,
- raises mean time-to-collision from `4.86 s` to `8.10 s`,
- lowers conflict-zone speed from `10.40 m/s` to `3.76 m/s`,
- lowers high-speed conflict-zone entry from `0.9875` to `0.2296`,
- raises creep-speed occupancy from `0.0000` to `0.6663`.

This is the behavior signature we wanted: the agent does not simply park, and it does not blast through. It probes at a low-moderate speed, survives longer when collisions still happen, and completes more episodes under denser traffic.

### What failed during shaping

Pure reward-only PPO from scratch learned a conservative local optimum: low collision exposure but poor success because it preferred waiting. A residual throttle prior made creeping visually obvious, but that was too hand-guided for this objective. The final notebook therefore keeps the action space unconstrained and uses curriculum distillation only to initialize the neural PPO actor. After distillation, the evaluated policy is the network itself.

### Why this is not a hard constraint

The final policy receives observations and outputs a continuous throttle action directly. The traffic-aware teacher is not used during evaluation or rendering. The reward and observation features teach the policy what negotiation means: TTC pressure, conflict-zone entry, slow probing, progress after a safe gap, and arrival. The learned policy can still choose aggressive or conservative throttle; it is not forced to follow a scripted speed profile.